# 🤖 LangChain + Bigdata MCP Integration

This notebook demonstrates how your AI agents can interact with **Bigdata.com via MCP (Model Context Protocol)**—a standardized way to connect AI applications to data sources and tools with automatic discovery.

## What This Demonstrates

This notebook demonstrate a cited, multi-source answer in one flow—combining internal portfolios and research with live market data, tearsheets, and calendars. **One MCP connection to Bigdata.com keeps your agents up to date with every new Bigdata.com capability without changing code.** The result: faster, traceable insights with automatic tool discovery and inline source links.

**Bigdata.com MCP Integration:**
- **Automatic Tool Discovery** → MCP exposes all available tools dynamically; when Bigdata.com adds new capabilities (tearsheets, calendars, screeners), your agent gets them automatically
- **Company Lookup** → Resolve tickers to entity IDs via Knowledge Graph
- **Search (`bigdata_search`)** → Query news, filings, transcripts, research, and private files. Supports two modes:
  - **`smart`** (default) → An AI agent interprets the natural-language query and infers entities, dates, sources, and content types for you
  - **`fast`** → A direct semantic + lexical search where you supply explicit filters (entity IDs, keywords, timestamps, categories, document types, sentiment)
- **Tearsheets** → Company and country financial profiles
- **Events Calendar** → Earnings dates, conference calls

**Internal Data Integration:**
- Connect to your portfolio databases (positions, transactions, P&L)
- Semantic search over internal research documents via vector stores
- Combine MCP-discovered tools with your internal tools seamlessly

**Framework Flexibility:**
> This demo uses **LangChain** with **langchain-mcp-adapters** and **LangSmith** for observability. The MCP protocol is framework-agnostic—**CrewAI**, **AutoGen**, **Google A2A**, and other frameworks can connect to MCP servers using their respective adapters. The key benefit: one integration, automatic access to all current and future Bigdata.com tools.

---

## What is MCP (Model Context Protocol)?

MCP is an open protocol that standardizes how AI applications connect to data sources and tools. Think of it as "USB for AI" - a universal connector.

**Key Benefits:**
- **Automatic Tool Discovery**: MCP servers expose tools dynamically - no manual updates needed when new tools are added
- **Standardized Interface**: One protocol works across all MCP-compatible tools
- **Stateful Connections**: Efficient communication with long-lived sessions

## Architecture

![Agent to Bigdata MCP](./static/agent-mcp.png)


---

## 1️⃣ Install Dependencies

Install from the project root before running this notebook:

```bash
uv sync
```

In [1]:
# Dependencies: install from project root with uv sync (see README)

## 2️⃣ Import Libraries

**LangChain**: ReAct agent and tool-calling

**LangChain MCP Adapters**: Bridge between LangChain and MCP servers

**langgraph_core** (reusable): Environment, `create_financial_database`, `create_vector_store`, local tools (`get_database_tools`, `get_vectorstore_tools`), and display helpers (`display_query`, `display_response`, `display_tools_used`, `display_citations`)

In [2]:
import os
import json
import sqlite3
import random
from datetime import datetime, timedelta
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from dotenv import load_dotenv

# Display utilities for Jupyter
from IPython.display import display, Markdown, HTML
import html as html_lib

# LangChain
from langchain.tools import tool
from langchain.agents import create_agent as langchain_create_agent
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# MCP integration
from langchain_mcp_adapters.client import MultiServerMCPClient

# Reusable core (langgraph_core): environment, data sources, display
import sys
sys.path.append(".")
from langgraph_core import (
    setup_environment,
    create_financial_database,
    create_vector_store,
    get_database_tools,
    get_vectorstore_tools,
    display_query,
    display_response,
    display_tools_used,
    display_citations,
)
load_dotenv()
print("✅ Libraries imported successfully")

/var/folders/7c/1g2h3ww906x15rwrst4sl0mc0000gp/T/ipykernel_95886/3566990655.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✅ Libraries imported successfully


In [3]:
# Environment, database, vector store, and display helpers are provided by langgraph_core
# (see langgraph_core.py). No local definitions needed for reusability.
print("✅ Using langgraph_core for environment, data sources, and display helpers")


✅ Using langgraph_core for environment, data sources, and display helpers


## 3️⃣ Setup Environment & Local Data Sources

Initialize:
- LangSmith tracing for observability
- Local SQLite database with sample portfolio data
- FAISS vector store with research documents

In [4]:
# Setup environment (loads API keys, enables LangSmith tracing)
config = setup_environment(
    langsmith_project="langgraph-bigdata-mcp-demo",
    enable_tracing=True
)

# Create local database with sample financial data
create_financial_database()

# Create vector store with research documents
create_vector_store()

print("\n✅ Local data sources ready")

✅ LangSmith tracing enabled → Project: langgraph-bigdata-mcp-demo
✅ Bigdata API Key: bd_v2_ONWV...
✅ OpenAI API Key: sk-proj--w...
✅ Created 3 accounts
✅ Created 3 portfolios
✅ Created 15 holdings
✅ Created 100 transactions
✅ Created vector store with 6 documents

✅ Local data sources ready


## 4️⃣ Load Local Tools

Load tools that interact with local data sources:

In [5]:
# Get local database tools
local_db_tools = get_database_tools()
print(f"✅ Loaded {len(local_db_tools)} database tools:")
for t in local_db_tools:
    print(f"   - {t.name}: {t.description[:80]}...")

# Get local vector store tools
local_vector_tools = get_vectorstore_tools()
print(f"\n✅ Loaded {len(local_vector_tools)} vector store tools:")
for t in local_vector_tools:
    print(f"   - {t.name}: {t.description[:80]}...")

✅ Loaded 2 database tools:
   - internal_query_database: Execute SQL query against the internal financial transactions database.

Availab...
   - internal_portfolio_summary: Get a summary of a specific portfolio from internal database including holdings ...

✅ Loaded 1 vector store tools:
   - internal_search_research: Search internal research documents using semantic similarity.

This searches thr...


## 5️⃣ Connect to Bigdata MCP Server

**MCP Configuration:**
- **URL**: `https://mcp.bigdata.com/`
- **Transport**: HTTP (streamable)
- **Authentication**: `x-api-key` header

The MCP client automatically discovers all available tools from the server.

In [6]:
# API keys
BIGDATA_API_KEY = os.getenv("BIGDATA_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not BIGDATA_API_KEY:
    raise ValueError("BIGDATA_API_KEY not found. Set via: export BIGDATA_API_KEY='your-key'")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found. Set via: export OPENAI_API_KEY='your-key'")

print(f"✅ Bigdata API Key: {BIGDATA_API_KEY[:10]}...")
print(f"✅ OpenAI API Key: {OPENAI_API_KEY[:10]}...")

✅ Bigdata API Key: bd_v2_ONWV...
✅ OpenAI API Key: sk-proj--w...


### Configure MCP Client

**Important**: Set `ANYIO_BACKEND='asyncio'` to ensure async backend detection in Jupyter:

In [7]:
# Set async backend for anyio (required in Jupyter)
os.environ['ANYIO_BACKEND'] = 'asyncio'

# Configure MCP client
mcp_client = MultiServerMCPClient(
    {
        "bigdata": {
            "url": "https://mcp.bigdata.com/",
            "transport": "http",  # Streamable HTTP transport
            "headers": {
                "x-api-key": BIGDATA_API_KEY
            }
        }
    }
)

print("✅ MCP client configured")

✅ MCP client configured


### Load MCP Tools

The MCP client automatically discovers all tools exposed by the Bigdata MCP server:

**Note**: In Jupyter notebooks, we use `await` directly instead of `asyncio.run()` because Jupyter already runs an event loop in the background.

In [8]:
# Load tools from Bigdata MCP server
# Note: In Jupyter notebooks, there's already a running event loop, so we use 'await' directly
print("Connecting to Bigdata MCP...")
bigdata_mcp_tools = await mcp_client.get_tools()
print(f"✅ Loaded {len(bigdata_mcp_tools)} tools from Bigdata MCP:")
for tool in bigdata_mcp_tools:
    print(f"   - {tool.name}: {tool.description[:80]}...")

Connecting to Bigdata MCP...
✅ Loaded 19 tools from Bigdata MCP:
   - bigdata_mcp_instructions: [SYSTEM CONTEXT - DO NOT CALL THIS TOOL - ONLY READ THE DESCRIPTION BELOW] Instr...
   - bigdata_search: Primary search tool for retrieving financial and business content from Bigdata's...
   - fetch: This tool returns the document by its ID.
...
   - bigdata_company_tearsheet: Returns a comprehensive company tearsheet with financial data, market intelligen...
   - bigdata_events_calendar: Returns a professionally formatted markdown calendar of corporate events includi...
   - bigdata_country_tearsheet: Returns a comprehensive country economic tearsheet with a sectoral macroeconomic...
   - bigdata_market_tearsheet: Returns a comprehensive market snapshot in markdown covering eight asset classes...
   - bigdata_etf_tearsheet: Returns a comprehensive ETF tearsheet in markdown covering fund facts, top holdi...
   - find_securities: **Routing (hard rule):** Use **`find_securities`** for **ETFs*

### 🔎 The updated `bigdata_search` tool — fast vs smart modes

`bigdata_search` is the primary retrieval tool. It searches SEC filings, earnings transcripts, news, broker/analyst research, podcasts, and your private uploaded files, returning **document chunks** with timestamps, source names, and URLs for citation.

It accepts a single `request` object with a `search_mode` and a `query`:

**Smart mode** (recommended default) — an AI agent interprets a natural-language query and infers entities, dates, sources, and content types. You do **not** need to resolve tickers first.

```json
{
  "search_mode": "smart",
  "query": {
    "text": "Netflix's partnership in news",
    "context": "search in news",   // optional scope/source/tag hints
    "max_chunks": 20
  }
}
```

**Fast mode** — a direct semantic + lexical search where you supply explicit `filters` for deterministic control:

```json
{
  "search_mode": "fast",
  "query": {
    "text": "Netflix's partnership",
    "max_chunks": 20,
    "filters": {
      "document_type": { "mode": "INCLUDE", "values": [ { "type": "NEWS" } ] }
    }
  }
}
```

**Fast-mode filters** (all optional, each supports `any_of` / `all_of` / `none_of`): `entity`, `keyword`, `timestamp` (`start`/`end` ISO 8601), `category` (`news`, `filings`, `transcripts`, `research`, `my_files`), `reporting_entities`, `reporting_periods`, `sentiment` (`positive`/`negative`/`neutral`), and `document_type` (`NEWS`, `FILING`, `TRANSCRIPT`, `INVESTMENT-RESEARCH`, ...).

> When the LangChain agent calls `bigdata_search`, the LLM builds this `request` object automatically. The next cell shows both modes invoked **directly** so you can see the exact payloads and responses.

Reference: [`bigdata_search` tool docs](https://docs.bigdata.com/mcp-reference/tools/bigdata-search)

In [9]:
# Grab the bigdata_search tool from the auto-discovered MCP tools
search_tool = next(t for t in bigdata_mcp_tools if t.name == "bigdata_search")


def preview_search(raw_result, max_docs: int = 3, chunk_chars: int = 220) -> None:
    """Pretty-print the first few documents returned by bigdata_search."""
    data = json.loads(raw_result) if isinstance(raw_result, str) else raw_result
    results = data.get("results", data) if isinstance(data, dict) else data
    for doc in results[:max_docs]:
        source = doc.get("source", {}).get("name", "Unknown source")
        headline = (doc.get("headline") or "").strip()[:100]
        timestamp = doc.get("timestamp", "")
        print(f"• [{source} — {timestamp}] {headline}")
        for chunk in doc.get("chunks", [])[:1]:
            print(f"    {chunk.get('text', '')[:chunk_chars].strip()}...")
    print()


# 1) SMART MODE — natural language; entities/dates/sources inferred automatically
print("🧠 SMART MODE: 'Netflix partnerships' (news)\n")
smart_result = await search_tool.ainvoke(
    {
        "request": {
            "search_mode": "smart",
            "query": {
                "text": "Netflix's recent partnership announcements",
                "context": "search in news",
                "max_chunks": 10,
            },
        }
    }
)
preview_search(smart_result)

# 2) FAST MODE — explicit filters for deterministic control
print("⚡ FAST MODE: same topic, restricted to NEWS document type\n")
fast_result = await search_tool.ainvoke(
    {
        "request": {
            "search_mode": "fast",
            "query": {
                "text": "Netflix partnership",
                "max_chunks": 10,
                "filters": {
                    "document_type": {
                        "mode": "INCLUDE",
                        "values": [{"type": "NEWS"}],
                    }
                },
            },
        }
    }
)
preview_search(fast_result)

🧠 SMART MODE: 'Netflix partnerships' (news)

• [Unknown source — ] 

⚡ FAST MODE: same topic, restricted to NEWS document type

• [Unknown source — ] 



## 6️⃣ Combine All Tools

Merge tools from all sources:

In [10]:
# Combine all tools
all_tools = local_db_tools + local_vector_tools + bigdata_mcp_tools

print(f"\n✅ Total tools available: {len(all_tools)}")
print(f"   - Local DB tools: {len(local_db_tools)}")
print(f"   - Local vector store tools: {len(local_vector_tools)}")
print(f"   - Bigdata MCP tools: {len(bigdata_mcp_tools)}")


✅ Total tools available: 22
   - Local DB tools: 2
   - Local vector store tools: 1
   - Bigdata MCP tools: 19


## 7️⃣ Create LangChain Agent

**LangChain ReAct Agent:**
- **Reasoning**: Plans which tools to use based on user query
- **Acting**: Executes tool calls and processes results
- **Iteration**: Continues until query is fully answered

Uses `langchain.agents.create_agent` (the current, non-deprecated API) which creates an agent with:
- Agent executor (LLM with tool calling capabilities)
- Tool registry (all available tools)
- System prompt (guides agent behavior)
- Streaming support (for real-time responses)

In [11]:
# Define system prompt for the agent
SYSTEM_PROMPT = """You are an intelligent financial research assistant with access to multiple data sources:

**External Data (Bigdata.com MCP):**
- Tools dynamically loaded from Bigdata MCP server
- Tools include news, prices, tear sheet, search, company lookup, and other capital markets capabilities

**Using `bigdata_search` (fast vs smart mode):**
- Prefer **smart mode** by default for natural-language / macro / thematic questions. Smart mode resolves entities, dates, sources, and content types on its own, so you usually do NOT need to call `find_companies` first.
- Use **fast mode** when you already have explicit filters (entity IDs, keywords, timestamps, categories such as `news`/`filings`/`transcripts`/`research`, document types, or sentiment) and want precise, deterministic control.
- Follow the search discipline: one focus, one entity group, and one time period per call. Split mixed topics into separate searches.

**Internal Data (Company Systems):**
- `internal_query_database` - Execute SQL queries on portfolio/transaction database
- `internal_portfolio_summary` - Get portfolio holdings and performance summary
- `internal_search_research` - Search internal investment research documents

Guidelines:
- Use appropriate tools based on the query
- For portfolio questions, use internal database tools
- For market intelligence, use Bigdata MCP tools
- Combine multiple sources for comprehensive analysis

**Citation format:** Use inline citations with the **source name as the link text** (not the raw URL). Format as markdown: [Source Name](url) or [1](url), [2](url) so the reader sees a clickable source name. Do not paste full URLs in the body.

**Do not add a separate "Sources" or "References" or "External sources" block at the end** when you have already used inline citations in the text. Inline citations are sufficient.
**Do not offer suggestions for follow up questions**

Available portfolios: PF001 (US Large Cap Growth), PF002 (AI & Semiconductor Focus), PF003 (Diversified Tech Leaders)
"""

model = "gpt-5"
# Initialize LLM
llm = ChatOpenAI(
    model=model,
    temperature=0,
    api_key=OPENAI_API_KEY
)

# Create agent using langchain.agents.create_agent (non-deprecated)
agent = langchain_create_agent(llm, all_tools, system_prompt=SYSTEM_PROMPT)

print(f"✅ Agent created with {len(all_tools)} tools")
print(f"   Model: {model}")
print(f"   System prompt configured")

✅ Agent created with 22 tools
   Model: gpt-5
   System prompt configured


## 8️⃣ Run Example Queries

Let's test the agent with queries that utilize different data sources:

### Example 1 : Multinode

In [12]:
query = """
Analyze the AI & Semiconductor Focus portfolio (PF002):
1. What are our current holdings and their performance?
2. What risks does our internal research identify?
3. For each holding, get us the pricing information from tearsheet
4. For each holding, get us negative news
"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
display_response(result)

# Display citations from Bigdata.com sources
#display_citations(result)

Here’s a consolidated view of PF002 (AI & Semiconductor Focus) across holdings, risks (from our internal research), current pricing (from tearsheets), and recent negative news.

1) Current holdings and performance (internal)
- NVIDIA (NVDA)
  - Shares: 12,000 | Avg cost: $450.00 | Current price: $875.50
  - Market value: $10,506,000 | Unrealized P&L: $5,106,000 (+94.6%)
  - Weight: 65.7%

- Broadcom (AVGO)
  - Shares: 1,500 | Avg cost: $850.00 | Current price: $1,425.00
  - Market value: $2,137,500 | Unrealized P&L: $862,500 (+67.6%)
  - Weight: 13.4%

- Palantir (PLTR)
  - Shares: 25,000 | Avg cost: $18.50 | Current price: $65.25
  - Market value: $1,631,250 | Unrealized P&L: $1,168,750 (+252.1%)
  - Weight: 10.2%

- AMD (AMD)
  - Shares: 8,000 | Avg cost: $95.00 | Current price: $145.25
  - Market value: $1,162,000 | Unrealized P&L: $402,000 (+52.9%)
  - Weight: 7.3%

- Taiwan Semi (TSM)
  - Shares: 3,000 | Avg cost: $110.00 | Current price: $185.75
  - Market value: $557,250 | Unrealized P&L: $227,250 (+68.9%)
  - Weight: 3.5%

Portfolio totals
- Total market value: $15,994,000
- Cost basis (implied): $8,227,500
- Unrealized P&L: $7,766,500
- Aggregate return on cost: +94.4%
- Holdings: 5

2) Risks highlighted in our internal research (by name/theme)
- Macro/valuation/AI-cycle
  - Valuation risk elevated for mega-cap tech and AI leaders; multiple compression possible if AI monetization lags expectations. AI capex could be front‑loaded relative to realized ROI. (Technology Sector Risk Assessment – Jan 2025)
- NVIDIA (NVDA)
  - China export restrictions exposure; supply constraints and execution risk around rapid product cadence. (NVIDIA Thesis Update – Dec 2024; Sector Risk Assessment – Jan 2025)
- AMD (AMD)
  - ROCm/software ecosystem still trails CUDA; execution risk vs. high valuation; competition in accelerators. (AMD AI Opportunity – Dec 2024; Strategy memo – Jan 2025)
- Palantir (PLTR)
  - Political/regulatory scrutiny and renewal risk for government deals; European sovereignty pushback; concentration in public sector. (Internal Strategy and risk assessments 2024–2025 corpus)
- TSMC (TSM)
  - Geopolitical concentration risk (Taiwan); capacity/energy/infrastructure constraints; policy/export‑control volatility. (Internal cross-name supply chain and macro notes 2024–2025)
- Portfolio-level hedging noted
  - Consider index put spreads/cash buffer during elevated AI enthusiasm phases. (Risk Assessment – Jan 2025)

3) Pricing snapshot from tearsheets (as of timestamps shown)
- NVIDIA (NVDA) — as of Jul 06, 2026 06:51 PM UTC
  - Price: $196.53 | 1D: +0.88% | 1M: -4.18% | 1Y: +24.20%
  - Market cap: $4,760.27B | 52W: $157.34–$236.54

- Broadcom (AVGO) — as of Jul 06, 2026 06:51 PM UTC
  - Price: $375.32 | 1D: +4.12% | 1M: -2.70% | 1Y: +36.89%
  - Market cap: $1,785.60B | 52W: $269.58–$495.00

- Palantir (PLTR) — as of Jul 06, 2026 06:51 PM UTC
  - Price: $132.65 | 1D: +2.59% | 1M: -2.12% | 1Y: -4.65%
  - Market cap: $304.57B | 52W: $106.37–$207.52

- AMD (AMD) — as of Jul 06, 2026 06:51 PM UTC
  - Price: $553.45 | 1D: +6.88% | 1M: +18.67% | 1Y: +310.57%
  - Market cap: $902.46B | 52W: $133.50–$584.73

- Taiwan Semiconductor (TSMC; TAI:2330) — as of Jul 06, 2026 05:30 AM UTC
  - Price: NT$2,460 | 1D: +0.61% | 1M: +4.02% | 1Y: +127.78%
  - Market cap: NT$63,793.70B | 52W: NT$1,060–NT$2,540

Data sources (tearsheets)
- Financial Modeling Prep via Bigdata.com

4) Negative news highlights (recent)
- NVIDIA (NVDA)
  - Reports allege manufacturing challenges could delay the Kyber NVL144 rack-scale system by 12+ months, weighing on sentiment and suppliers; Nvidia says its roadmap is intact. [International Business Times - Jul 06, 2026](https://app.bigdata.com/documents/6A4F4509AE05AD9FC5079321CDF89700?cnum=1), [Yahoo Finance France - Jul 06, 2026](https://app.bigdata.com/documents/9103E1F8579A69B8AC8A45C32FF0BA0E?cnum=2), [Sina - Jul 06, 2026](https://app.bigdata.com/documents/56BA35CEADC7873328CFC9EB05F0D508?cnum=2)
  - Valuation/expectations risk cited amid mixed AI sentiment and custom silicon competition. [Kalkine Media - Jul 06, 2026](https://app.bigdata.com/documents/A2192AD161A0DE882C9D216B2D39AACB?cnum=7)

- Broadcom (AVGO)
  - Post-earnings selloff: guidance and margin trajectory underwhelmed elevated expectations; concern over customer concentration and AI mix margin pressure. [Nasdaq - Jul 06, 2026](https://app.bigdata.com/documents/FA52DFC39BFA94752126BB636729D771?cnum=1), [Benzinga - Jun 05, 2026](https://app.bigdata.com/documents/FAD53997AAE527F3DF8B3D35A975B976?cnum=1), [Market Screener - Jun 04, 2026](https://app.bigdata.com/documents/97C39136C8FC9684C8AEC282DFE1FD51?cnum=1)
  - Supply constraints and potential TSMC capacity crunch flagged. [MT Newswires - Mar 24, 2026](https://app.bigdata.com/documents/A30AFFB808B3596FEE655C227462E734?cnum=1)

- Palantir (PLTR)
  - UK NHS contract faces review and political scrutiny; concerns around vendor lock‑in and sensitive public‑sector reliance. [MT Newswires - Jun 09, 2026](https://app.bigdata.com/documents/B64BA106452561F2EDB47E89B3A82243?cnum=1), [MT Newswires - Jun 03, 2026](https://app.bigdata.com/documents/C9464B3DED034F03D2E90AF22973E713?cnum=1), [Financial Times - Jul 03, 2026](https://app.bigdata.com/documents/4F8540687D347EF960315FF9F1E9449F?cnum=1)
  - European pushback: legal setback in Switzerland; reported contract loss in France; broader sovereignty concerns. [Financial Times - Jun 12, 2026](https://app.bigdata.com/documents/4F83459871803E7FA00F762BA96F5B8D?cnum=2), [Alliance News - Jun 16, 2026](https://app.bigdata.com/documents/874E9961EE6B430132A102EC7B7A7588?cnum=2)

- AMD (AMD)
  - Debate on AI share gains: downgrades/notes flag weaker-than-expected AI GPU demand vs. Nvidia and execution risk; stock volatility around results. [Nasdaq - Jun 04, 2026](https://app.bigdata.com/documents/F202127BB47E2ED929E0F06CB4F94291?cnum=2), [Crypto Wire - Feb 04, 2026](https://app.bigdata.com/documents/6DA35D21E9B55012DC63A631BF455E83?cnum=4), [Financial Times - Feb 03, 2026](https://app.bigdata.com/documents/005B4808530559401236D061F01429C3?cnum=3)
  - CPU supply tightness and China delivery delays amid AI-driven component shortages. [Benzinga - Feb 06, 2026](https://app.bigdata.com/documents/66B93D152D122C2EDA58F3AF0613F43C?cnum=1)

- Taiwan Semiconductor (TSMC)
  - AI demand straining supply chain and equipment availability; capacity tight. [MT Newswires - Asia Pacific - Jun 05, 2026](https://app.bigdata.com/documents/A04AF0B2BFF32B6654ABA4EB6AC2060A?cnum=1)
  - Policy/geopolitical risks: potential tighter export controls to China; energy/logistics vulnerabilities cited. [Yahoo! Finance - Jun 09, 2026](https://app.bigdata.com/documents/77D1DEA6F5B814B936FF1097EC64305E?cnum=3), [Nasdaq - Apr 27, 2026](https://app.bigdata.com/documents/EA121984CE824AE8B8924E22422394B5?cnum=4), [MT Newswires - Asia Pacific - Apr 17, 2026](https://app.bigdata.com/documents/A9C62C92F44F6885729A8722826DEAE1?cnum=1)

If you want, I can export this as a memo and save it to your Bigdata.com workspace with tags (e.g., topic:pf002, date:2026-07-06).

### Example 2: Bigdata MCP Tool Query

Use external market intelligence:

In [13]:
# Example 2: Use Bigdata MCP tools for external data
query = "Find the latest news about NVIDIA's earnings and revenue growth using Bigdata tools."

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
#display_response(result)

# Display citations from Bigdata.com sources
#display_citations(result)

In [14]:
# Display response
display_response(result)

# Display citations from Bigdata.com sources
display_citations(result)

Here’s the latest on NVIDIA’s earnings and revenue growth (sourced via Bigdata.com):

- Latest quarterly results (Q1 FY2027, reported May 2026): Revenue came in at $81.6B, up about 85% year over year, with non-GAAP EPS of $1.87 (+140% y/y) and gross margin around 75%. Data Center revenue rose to roughly $75.2B (+92% y/y) [Nasdaq - Jul 06, 2026](https://app.bigdata.com/documents/F32BCA0F314425840D679224D49204EA?cnum=4&cnum=5), [FinanceFeeds - Jul 06, 2026](https://app.bigdata.com/documents/96F102F60CD028669BEAA5A12F6A6151?cnum=14).

- Guidance points to continued acceleration: Management guided Q2 FY2027 revenue to about $91B (±2%), implying roughly 95% y/y growth and ~11.5% sequential growth; gross margin guidance remains around 75% [Nasdaq - Jul 06, 2026](https://app.bigdata.com/documents/F32BCA0F314425840D679224D49204EA?cnum=4&cnum=5), [FinanceFeeds - Jul 06, 2026](https://app.bigdata.com/documents/96F102F60CD028669BEAA5A12F6A6151?cnum=14), [The Globe And Mail - Jul 06, 2026](https://app.bigdata.com/documents/CF6E14312BBCB2F0E965302091142475?cnum=2).

- Cash flow supporting growth and buybacks: In Q1 FY2027, operating cash flow was $50.3B and free cash flow $48.6B. NVIDIA raised its quarterly dividend to $0.25 and authorized an additional $80B in repurchases, bringing total remaining buyback capacity to about $119B [Nasdaq - Jul 06, 2026](https://app.bigdata.com/documents/38FB08A751308C6EA1B1FBBD290A457E?cnum=2&cnum=1).

- Industry-relative growth remains exceptional: Recent coverage highlights NVIDIA’s revenue growth around 85% y/y—well above many semiconductor peers—alongside strong profitability metrics (ROE, gross profit, EBITDA) [Benzinga - Jul 06, 2026](https://app.bigdata.com/documents/BEC79DC1ABBB89B439A8CD319ACDBC9D?cnum=3&cnum=2&cnum=4).

### Example 3: Multi-Source Query

Combine local and external data:

In [15]:
# Example 3: Multi-source comprehensive analysis
query = """For our NVIDIA holdings:
1. Check our internal database to see which portfolios hold NVDA and how much
2. Search our internal research for our investment thesis
3. Use Bigdata tools to find recent news about NVIDIA
4. Provide a comprehensive summary combining all sources"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
display_response(result)

# Display citations from Bigdata.com sources
#display_citations(result)

Here’s a consolidated view of our NVIDIA exposure, our internal thesis, and the latest external developments.

1) Our NVIDIA holdings (internal database)
- PF002 — AI & Semiconductor Focus
  - Position: 12,000 shares NVDA
  - Avg cost: $450.00; current price (db): $875.50
  - Market value: $10.506m
  - Unrealized P&L: +$5.106m
  - Portfolio AUM: $15.0m → NVDA weight ≈ 70.0%

- PF003 — Diversified Tech Leaders
  - Position: 8,000 shares NVDA
  - Avg cost: $520.00; current price (db): $875.50
  - Market value: $7.004m
  - Unrealized P&L: +$2.844m
  - Portfolio AUM: $50.0m → NVDA weight ≈ 14.0%

- Total across portfolios
  - Shares: 20,000; Market value: $17.510m; Unrealized P&L: +$7.950m

2) Internal research highlights (investment thesis and risk lens)
- Core thesis (Dec 15, 2024 thesis update): NVIDIA remains top pick in semis on surging AI data center demand (H100/H200) with next-gen Blackwell B100/B200 expected to drive another performance step-up; durable CUDA software moat (4M+ developers) and large AI inference TAM by 2027; Strong Buy with $950 PT (25x FY26E EPS).
- Strategy (Jan 5, 2025 memo): Recommended +3% portfolio weight increase in NVDA on sustained AI training demand outpacing supply.
- Key risks (Jan 10, 2025 tech risk assessment): Valuation compression risk; regulatory/export-control exposure (China 20–25% revenue at risk); competition (AMD MI300/custom silicon); supply constraints; potential mismatch if AI spend pulls forward ahead of realized ROI.

3) Recent external developments on NVIDIA (Bigdata.com)
- Roadmap and timing
  - NVIDIA told Bloomberg its “roadmap is intact,” disputing a SemiAnalysis note claiming delay to a future Kyber/Rubin architecture; pushback suggests management confidence in execution despite manufacturing complexity at the rack/system level [The Fly - Jul 06, 2026](https://app.bigdata.com/documents/E0DDFA02A9E0243FFCF6E3C78B22345D?cnum=1). Coverage also summarized SemiAnalysis’ manufacturability concerns (e.g., 78-layer PCB midplane) and NVIDIA’s denial that timelines are slipping [Crypto Briefing - Jul 06, 2026](https://app.bigdata.com/documents/6CBB9EB0293BEF8EAC5A7598AE896FC7?cnum=1).

- Customer/partner and deployment signals
  - Google Cloud added confidential VMs based on NVIDIA Blackwell GPUs to its Confidential Computing portfolio, alongside security tooling upgrades—indicative of enterprise-grade adoption avenues for Blackwell [IT House - Jul 06, 2026](https://app.bigdata.com/documents/83F5CABC6B392C6E1FBAC11E4BC46CAD?cnum=1).
  - CoreWeave highlighted Blackwell and Rubin platforms in renewable-powered Swedish data centers, interconnected with Quantum‑X800 InfiniBand, pointing to continued acceleration in high-density AI infrastructure builds [Energy Digital - Jul 01, 2026](https://app.bigdata.com/documents/5A84C9358894B9480C0C363AAFE3976B?cnum=1).

- Regulatory/export-control backdrop and supply chain enforcement
  - Taiwan regulators are tightening AI server export enforcement; discussion underway on broader curbs that could impact diversion risk for AI servers using NVIDIA chips [International Business Times - Jun 30, 2026](https://app.bigdata.com/documents/F5A460FFB37221A511D30F64405CA69D?cnum=5).
  - Super Micro reported staff detentions in an AI server smuggling probe in Taiwan, underscoring heightened enforcement risk in key assembly hubs [Crypto Briefing - Jul 02, 2026](https://app.bigdata.com/documents/494D9693420BAB30B638C569AF5578A3?cnum=2).
  - In the US, some lawmakers continue to push stricter chip/AI export controls; additional rounds could force further China‑specific product redesigns for NVIDIA [Crypto Briefing - Jul 05, 2026](https://app.bigdata.com/documents/14CC59CE70980F3059DFCB14C06D97D2?cnum=2).

- Market narrative and flows
  - Strategists highlight a potential rotation from chipmakers to hyperscalers amid valuation concerns for semis; near-term returns could depend on hyperscaler capex signals and whether AI spend expectations are tempered [Yahoo! Finance - Jul 06, 2026](https://app.bigdata.com/documents/E7BBCAB8EBB8B10E8EA8D91C5EE87158?cnum=2).

4) Synthesis and implications
- Positioning and concentration
  - PF002’s NVDA weight (~70%) is very high for a single name, even in a focused strategy; PF003’s 14% weighting is material but more balanced. Mark-to-db pricing implies significant embedded gains (+$7.95m aggregate unrealized P&L). The exposure strongly ties portfolio outcomes to NVIDIA’s Blackwell ramp, networking (NVLink/InfiniBand/Ethernet), and hyperscaler capex trajectories.

- Thesis alignment vs new information
  - The internal thesis around CUDA moat, AI training/inference scale-up, and Blackwell performance uplift remains broadly supported by external updates: Google Cloud’s Blackwell confidential VMs and CoreWeave deployments add real-world adoption data points.
  - Roadmap risk is an active watch item: while NVIDIA publicly denies delays, third‑party reports flag manufacturability challenges at system/rack scale. Execution on Kyber/Rubin-era architectures is critical—slippage could open windows for AMD/custom silicon. Both sides of the story are present; management’s denial reduces but does not eliminate timeline risk [The Fly - Jul 06, 2026](https://app.bigdata.com/documents/E0DDFA02A9E0243FFCF6E3C78B22345D?cnum=1), [Crypto Briefing - Jul 06, 2026](https://app.bigdata.com/documents/6CBB9EB0293BEF8EAC5A7598AE896FC7?cnum=1).
  - Regulatory/export dynamics are incrementally tighter: Taiwan’s enforcement and US pressure raise the probability of further friction in China-directed SKUs and server flows—consistent with our internal risk flag on China exposure and export controls [International Business Times - Jun 30, 2026](https://app.bigdata.com/documents/F5A460FFB37221A511D30F64405CA69D?cnum=5), [Crypto Briefing - Jul 05, 2026](https://app.bigdata.com/documents/14CC59CE70980F3059D?cnum=2).
  - Market technicals/flows: the suggested rotation away from semis implies near-term factor/positioning risk for a crowded winner; outcomes hinge on upcoming hyperscaler capex updates and supply ramp cadence [Yahoo! Finance - Jul 06, 2026](https://app.bigdata.com/documents/E7BBCAB8EBB8B10E8EA8D91C5EE87158?cnum=2).

- Portfolio risk management considerations
  - PF002’s concentration elevates idiosyncratic roadmap and policy risk; consider whether to moderate exposure tactically or overlay hedges around event windows (e.g., earnings/roadmap milestones, US/Taiwan export actions).
  - Key monitoring items: Blackwell volume ramp and lead times; NVIDIA networking (NVLink/InfiniBand/Ethernet) growth vs compute; hyperscaler capex mix and any softening in plans; any China‑compliant SKU changes; and competitor traction (AMD MI300 and custom silicon attach) relative to CUDA stickiness.

If you’d like, I can set up automated monitoring for: NVIDIA roadmap headlines, hyperscaler capex commentary, Taiwan/US export-control actions, and any Blackwell deployment announcements—so we can react quickly within PF002 and PF003.

### Example 4: Company Briefing 

In [16]:
query = """
Summarize recent developments for CoreWeave (last 30 days).
**Steps:**
1. Call find_companies and get the company id
2. Call bigdata_tearsheet and get business context
3. Use bigdata_search and find news in the last 30 days
4. Categorize findings
**Categories:**
- Financial results
- Product/tech launches
- M&A and partnerships
- Regulatory/legal updates
- Management changes
- Other material events
For each: Date, facts, investment implications (bullish/bearish/neutral).
Please add inline source attribution.
"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)



In [17]:
# Display response
display_response(result)

# Display citations from Bigdata.com sources
#display_citations(result)

Here’s a concise, categorized wrap-up of CoreWeave developments over the last 30 days, with dates, facts, and investment implications.

Business context (for orientation)
- CoreWeave is a specialized AI cloud provider focused on high-performance GPU infrastructure for training and inference, expanding capacity rapidly to meet GenAI demand (per company overview and recent fundamentals from the Bigdata.com tearsheet).

Financial results / capital markets
- Jun 11, 2026 — Raised ~US$3.5B via first euro- and dollar-denominated high-yield bonds to accelerate global AI data center buildout. Implication: bullish for capacity expansion and funding flexibility; watch leverage/interest-cost risk (neutral to mildly bearish on balance sheet). [Yahoo! Finance - Jun 11, 2026](https://app.bigdata.com/documents/486F922AE6A48B56739931A067C8D7F9?cnum=1)

Product/tech launches
- Jun 24, 2026 — Co-location agreement with Conapto to add immediate AI cloud capacity in Stockholm, powered by renewable energy and NVIDIA Blackwell/Vera Rubin platforms with Quantum‑X800 InfiniBand. Implication: bullish for EU footprint, sustainability profile, and time-to-serve for enterprise AI workloads. [PubT - Jun 24, 2026](https://app.bigdata.com/documents/F98C482A2F7275EBFDE7361FFC7F0D1C?cnum=2)
- Jul 6, 2026 — Named a Visionary in the 2026 Gartner Magic Quadrant for Cloud AI Infrastructure (company announcement). Implication: bullish for credibility with enterprise buyers navigating AI-cloud vendor selection. [PubT - Jul 6, 2026](https://app.bigdata.com/documents/5393DE3D7533008D925BD8E9778349F6?cnum=1)

M&A and partnerships
- Jun 23, 2026 — Five-year, multi‑exabyte, $335M storage agreement with Backblaze to expand CoreWeave AI Object Storage tiers and preserve high-performance storage for AI workloads. Implication: bullish for cost-efficient storage scaling, service tiering, and margin mix on storage. [Backblaze Inc. (press release) - Jun 23, 2026](https://app.bigdata.com/documents/8733AE16BCFD5811B4E2FA9D5F641769?cnum=1)

Regulatory/legal updates
- None material identified in the last 30 days.

Management changes
- Jul 6, 2026 (filed for Jun 30) — Insider transaction: CSO/Director Brian Venturo sold 142,405 shares (~$12.96M). Implication: neutral-to-bearish optics; not necessarily thesis-changing given ongoing capex cycle and insider diversification possibilities. [MT Newswires - Jul 6, 2026](https://app.bigdata.com/documents/2FEDFFAE208B019E0FEF930159D999D8?cnum=1)

Other material events
- Jun 12–15, 2026 — Added to Nasdaq‑100 (effective Jun 22). Implication: bullish for index-driven demand/liquidity and institutional visibility; near-term technical volatility around inclusion noted. [Benzinga - Jun 12, 2026](https://app.bigdata.com/documents/3A8DF8E7C2BB7E2772613AA77647D3C8?cnum=1), [Yahoo! Finance - Jun 15, 2026](https://app.bigdata.com/documents/FB76534FC7F3B89CDCDC1BE934A7E960?cnum=1)
- Jul 1–2, 2026 — Competitive overhang: Reports Meta is developing an external AI cloud/computing business; CRWV sold off. Bernstein flagged the potential move as “problematic” for CoreWeave; Rosenblatt called the selloff a buying opportunity, citing persistent GPU tightness and contract constraints on Meta reselling leased capacity. Implication: mixed—bearish on potential pricing/competition risk; partially offset by demand/supply tightness and analyst support (net neutral to mildly bearish near term, watch follow-through). [Benzinga - Jul 1, 2026](https://app.bigdata.com/documents/C9DADB45E7A33F876E08C9BB0A32EE3D?cnum=1), [The Fly (Bernstein) - Jul 1, 2026](https://app.bigdata.com/documents/F281124693E3958EF222FC4B1F80EEE3?cnum=1), [The Fly (Rosenblatt) - Jul 2, 2026](https://app.bigdata.com/documents/DCE92ED79761A75C1E1A8E4F9C5DD619?cnum=1), [Benzinga - Jul 2, 2026](https://app.bigdata.com/documents/391FFEB993095E720C2A7EB3BDC8DC54?cnum=1), [The Globe And Mail - Jul 5, 2026](https://app.bigdata.com/documents/408CCE60E9221328754C560EEA5A4444?cnum=1)
- Jun 24, 2026 — Analyst initiation: Rosenblatt started CRWV at Buy with $250 PT, highlighting scale (~50 data centers) and positioning as a leading AI compute provider. Implication: bullish for sentiment and coverage breadth. [The Fly - Jun 24, 2026](https://app.bigdata.com/documents/3D9E70C7024A173610FA78771C2B3BBD?cnum=1)
- Jul 6, 2026 — Sell-side commentary roundup: Northland reaffirmed Buy ($165 PT); BNP Paribas cited strong AI-capacity demand and improving execution H2’26 after recent share underperformance. Implication: bullish for medium-term sentiment despite recent volatility. [Yahoo! Finance - Jul 6, 2026](https://app.bigdata.com/documents/959A863D24D4F94B536D22DD075754D2?cnum=1&cnum=2)

Net takeaways
- Demand/capacity: Expansion in Europe (Conapto) and third-party storage (Backblaze) underscore continued scaling; positive for service breadth and time-to-revenue. Bullish.
- Funding: High-yield access at size suggests strong market appetite to finance buildout; leverage remains the key risk to watch. Mixed (strategically bullish; balance-sheet risk).
- Competitive landscape: Meta’s potential entry spotlighted competitive/pricing overhangs; however, multiple checks still point to GPU tightness and large backlogs. Mixed-to-cautious near term.
- Market positioning: Gartner Visionary recognition and Nasdaq‑100 inclusion aid credibility and shareholder base depth. Bullish.

### Example 5: Local Database Query

Query internal portfolio holdings:

In [18]:
# Example 5: Query internal database for top holdings
query = "What are our top 5 holdings by market value across all portfolios?"

# Display query
display_query(query)

# Run agent (async invocation in Jupyter)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
display_response(result)


Top 5 holdings by total market value across all portfolios (USD):
- NVDA — NVIDIA Corporation: $17,510,000 (~34.9% of total)
- MSFT — Microsoft Corporation: $9,556,500 (~19.0%)
- AAPL — Apple Inc.: $7,410,000 (~14.8%)
- CRM — Salesforce Inc.: $3,255,000 (~6.5%)
- META — Meta Platforms Inc.: $2,632,500 (~5.2%)

Notes:
- Currency: USD (all accounts)
- Total portfolio market value: $50,185,750
- Top 5 represent ~80% of total market value across PF001, PF002, and PF003.

### Example 6: Local Vector Store Query

Search internal research documents:

In [19]:
# Example 6: Search internal vector store
query = "What does our internal research say about NVIDIA's competitive moat?"

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
display_response(result)


Here’s the distilled view from our internal research on NVIDIA’s competitive moat:

What makes the moat strong
- Software and ecosystem lock-in: CUDA’s vast installed base (4M+ developers) and rich libraries/tooling create high switching costs and developer mindshare that competitors (notably AMD’s ROCm) still trail meaningfully. (NVIDIA Q4 2024 Investment Thesis Update, 2024-12-15; AMD – Data Center & AI Opportunity Assessment, 2024-12-11)
- Silicon cadence and performance leadership: Rapid architecture cycles (Hopper → Blackwell) with step-function gains (Blackwell flagged at ~2.5x over prior gen) sustain a performance/watt and TCO edge that matters for both training and increasingly inference. (NVIDIA Q4 2024 Investment Thesis Update, 2024-12-15)
- Full-stack platform advantages: Tight integration across GPUs, high-bandwidth memory, interconnect (InfiniBand/NVLink), systems (DGX), and software runtime yields out-of-the-box performance and easier cluster scaling—hard for piecemeal rivals to replicate. (NVIDIA Q4 2024 Investment Thesis Update, 2024-12-15)
- Demand flywheel and share allocation: AI training demand well in excess of supply reinforces NVIDIA’s priority position with foundry and HBM suppliers, sustaining unit/pricing power and reinforcing the ecosystem lead. (Q1 2025 Portfolio Strategy – Technology Sector Allocation, 2025-01-05)

Where the moat could be challenged
- Improving competition: AMD’s MI300X is competitive on memory capacity and inference on some LLM workloads, and is landing marquee cloud deployments; the main gap remains software (ROCm vs CUDA). Custom accelerators (e.g., cloud TPUs/ASICs) are a persistent long-term threat in specific workloads. (AMD – Data Center & AI Opportunity Assessment, 2024-12-11)
- Geopolitical/regulatory risk: China export controls put 20–25% of revenue at risk and could drive product segmentation or dampen growth in a key region. (Technology Sector Risk Assessment, 2025-01-10)
- Supply-chain constraints: Tight HBM and advanced node capacity can bottleneck shipments and elongate lead times, potentially limiting upside and ceding openings to rivals if allocations shift. (NVIDIA Q4 2024 Investment Thesis Update, 2024-12-15)
- AI cycle/demand normalization: If enterprise AI ROI lags expectations, infrastructure spend could cool, pressuring pricing power and growth multiples. (Technology Sector Risk Assessment, 2025-01-10)

Portfolio positioning signal
- Our internal allocation raised NVDA by +3% (Q1 2025), reflecting conviction that near-term demand and the software/platform moat remain underappreciated relative to risks. (Q1 2025 Portfolio Strategy – Technology Sector Allocation, 2025-01-05)

Bottom line
- Our research views NVIDIA’s moat as primarily software- and ecosystem-led, reinforced by a fast silicon cadence and end-to-end platform integration. We see it as durable over the next 12–24 months, while monitoring ROCm maturation, custom silicon adoption, China exposure, and supply tightness as the key vectors that could narrow the gap.

Internal sources referenced
- NVIDIA Q4 2024 Investment Thesis Update (2024-12-15)
- AMD – Data Center & AI Opportunity Assessment (2024-12-11)
- Technology Sector Risk Assessment – January 2025 (2025-01-10)
- Q1 2025 Portfolio Strategy – Technology Sector Allocation (2025-01-05)

### Example 7: Macro / FX Outlook — Smart Search

A natural-language, macro-level question is a perfect fit for **`bigdata_search` smart mode**: there is no single ticker to resolve, and the search agent infers the relevant entities (INR, RBI, importers/exporters), sources, and recent time window on its own.

In [20]:
# Example 7: Macro/FX outlook using bigdata_search in smart mode
query = """
What is the market outlook on the Indian Rupee (INR)?
"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
display_response(result)

# Display citations from Bigdata.com sources
#display_citations(result)

Here’s a concise read on INR’s outlook, based on the latest market commentary and research:

What’s priced now
- Spot/range: USD/INR is hovering near 95.2–95.4, with traders expecting a near-term 94.8–95.8 range as the pair tracks the dollar and flows; importer USD demand and NDF-related settlements are adding intermittent pressure [Yahoo! Finance - Jul 06, 2026](https://app.bigdata.com/documents/3C53CFC322E5D1B55956C43840093ED2?cnum=1), [The Middle East: International Edition - Jul 06, 2026](https://app.bigdata.com/documents/8B02C2DD2D4BD84B82D5E6120189BA42?cnum=4), [Business Standard - Jul 03, 2026](https://app.bigdata.com/documents/2C1622941839A8005A0DF6A28097D155?cnum=1).
- RBI stance: The RBI is selectively intervening to smooth volatility and has previously tightened speculative positioning; policy rates are on hold as authorities balance inflation and growth risks [Alliance News - Apr 08, 2026](https://app.bigdata.com/documents/B7A039BCA67BBA86A41A76A39F6B7704?cnum=2).

Near-term drivers and bias
- Oil: Lower crude after progress on US–Iran talks has eased India’s import bill and supported INR; renewed Middle East risk would be a headwind [FXStreet News - Jul 02, 2026](https://app.bigdata.com/documents/AF30940E914954C87EADE487DB57E364?cnum=1), [MT Newswires - Jun 15, 2026](https://app.bigdata.com/documents/864CA939036EC7AD783E167BB542196D?cnum=1).
- US yields/Fed path: INR softens when US yields firm and the dollar rebounds; upcoming US data and FOMC minutes are key near-term catalysts [FXStreet News - Jul 01, 2026](https://app.bigdata.com/documents/CD382A1F5C81E1BA9907375C92D1433C?cnum=1&cnum=6), [FXStreet News - Jun 30, 2026](https://app.bigdata.com/documents/56A80092D605F655413BC0721D55BDBA?cnum=1), [FXStreet News - Jul 06, 2026](https://app.bigdata.com/documents/50C270896768C1384985E7F873FE9B35?cnum=1&cnum=5&cnum=2).
- Capital flows/policy supports: Authorities are courting foreign inflows (e.g., scrapping capital gains tax for foreign bond buyers; easing access for overseas investors), which is supportive for INR and local bonds; watch index-inclusion headlines and bond buying trends [Financial Times - Jun 05, 2026](https://app.bigdata.com/documents/46357C8FF55D80BF85E2A910922D528E?cnum=2), [Alliance News - Jun 05, 2026](https://app.bigdata.com/documents/10B50B54157679683AB2DF377C75A2C6?cnum=3), [Yahoo! Finance - Jul 06, 2026](https://app.bigdata.com/documents/3C53CFC322E5D1B55956C43840093ED2?cnum=1).
- Flow/technical dynamics: Exporter activity, FII equity flows, and NDF maturities are shaping intra-week swings; recent DXY dips and softer oil have helped INR stabilize from June’s pressure [Yahoo! Finance - Jul 06, 2026](https://app.bigdata.com/documents/3C53CFC322E5D1B55956C43840093ED2?cnum=1), [Idbi - Jul 03, 2026](https://app.bigdata.com/documents/3976DD1AF0B1F26AE3774AE81B66E4B2?cnum=12), [Business Standard - Jul 03, 2026](https://app.bigdata.com/documents/2C1622941839A8005A0DF6A28097D155?cnum=1).

Base case
- Range-bound with a mild stabilization bias if oil stays contained and the Fed message doesn’t re-ignite a broad USD rally. RBI intervention and policy measures should limit disorderly moves, but persistent importer USD demand can cap INR gains near-term [Yahoo! Finance - Jul 06, 2026](https://app.bigdata.com/documents/3C53CFC322E5D1B55956C43840093ED2?cnum=1), [Business Standard - Jul 03, 2026](https://app.bigdata.com/documents/2C1622941839A8005A0DF6A28097D155?cnum=1), [FXStreet News - Jul 06, 2026](https://app.bigdata.com/documents/50C270896768C1384985E7F873FE9B35?cnum=1&cnum=5&cnum=2).

Key risks
- Upside risks to USD/INR (weaker INR): Oil spike from geopolitical flare-ups; renewed US yield strength; sustained FII equity outflows; monsoon-related inflation shocks; budget slippage widening the external gap [Financial Times - Jun 05, 2026](https://app.bigdata.com/documents/46357C8FF55D80BF85E2A910922D528E?cnum=2), [Idbi - Jul 03, 2026](https://app.bigdata.com/documents/3976DD1AF0B1F26AE3774AE81B66E4B2?cnum=12), [Tekedia (PubT) - May 25, 2026](https://app.bigdata.com/documents/4AF95559380F376A68E93809D3A4BCFC?cnum=5&cnum=3).
- Downside risks to USD/INR (stronger INR): Durable crude softness; clearer Fed pause signaling; acceleration of foreign bond inflows and index-inclusion momentum; continued RBI smoothing [MT Newswires - Jun 15, 2026](https://app.bigdata.com/documents/864CA939036EC7AD783E167BB542196D?cnum=1), [Yahoo! Finance - Jul 06, 2026](https://app.bigdata.com/documents/3C53CFC322E5D1B55956C43840093ED2?cnum=1), [Financial Times - Jun 05, 2026](https://app.bigdata.com/documents/46357C8FF55D80BF85E2A910922D528E?cnum=2).

Medium-term perspective
- Gradual depreciation trend over 2027–2030, with the RBI using reserves and selective tools to curb excess volatility; manageable CAD and improving business environment help anchor stability. Rupee internationalization efforts (bilateral settlements) provide only gradual structural support over time [The Economist - Jun 04, 2026](https://app.bigdata.com/documents/4E3B0C95F34138A485CB0F234AFCDD63?cnum=150).

Upcoming catalysts to watch
- US FOMC minutes and labor data (NFP) for guidance on the USD and global yields; oil headlines tied to Middle East developments; India bond flow/benchmark inclusion updates [FXStreet News - Jun 30, 2026](https://app.bigdata.com/documents/56A80092D605F655413BC0721D55BDBA?cnum=1), [Yahoo! Finance - Jul 06, 2026](https://app.bigdata.com/documents/3C53CFC322E5D1B55956C43840093ED2?cnum=1), [FXStreet News - Jul 06, 2026](https://app.bigdata.com/documents/50C270896768C1384985E7F873FE9B35?cnum=1&cnum=5&cnum=2).

Bottom line: In the near term, USD/INR looks range-bound and headline-driven, with RBI smoothing and recent policy moves cushioning downside risks to INR, while oil spikes and a firmer USD remain the primary threats to stability [Yahoo! Finance - Jul 06, 2026](https://app.bigdata.com/documents/3C53CFC322E5D1B55956C43840093ED2?cnum=1), [Financial Times - Jun 05, 2026](https://app.bigdata.com/documents/46357C8FF55D80BF85E2A910922D528E?cnum=2), [Business Standard - Jul 03, 2026](https://app.bigdata.com/documents/2C1622941839A8005A0DF6A28097D155?cnum=1).

## 9️⃣ Understanding the Agent Flow

The agent follows this process:

1. **Parse Query** → Understand what information is needed
2. **Plan** → Decide which tools to use
3. **Execute** → Call selected tools in sequence or parallel
4. **Synthesize** → Combine results into coherent answer
5. **Iterate** → If more information needed, repeat steps 2-4

**Tool Selection Logic:**
- Portfolio/holdings questions → `internal_query_database` or `internal_portfolio_summary`
- Internal research → `internal_search_research`
- External market data → Bigdata MCP tools (automatically discovered)

**Trace Visibility:**
- All tool calls are logged to LangSmith for debugging
- View traces at: https://smith.langchain.com

## 🔟 Key Benefits of This Architecture

### 1. **Automatic Tool Discovery**
- MCP server exposes tools dynamically
- No code changes needed when Bigdata.com adds new tools
- Agent automatically learns about new capabilities

### 2. **Unified Interface**
- Single agent interface for all data sources
- Consistent tool calling pattern
- Easy to add more MCP servers or local tools

### 3. **Stateful Reasoning**
- LangGraph maintains conversation state
- Agent can reference previous tool results
- Multi-turn reasoning supported

### 4. **Observability**
- LangSmith tracing shows full execution flow
- Easy to debug tool selection and results
- Performance monitoring built-in

## 🎯 Next Steps

**Extend this architecture:**

1. **Add More MCP Servers**
   ```python
   mcp_client = MultiServerMCPClient({
       "bigdata": {...},
       "other_mcp_server": {...}
   })
   ```

2. **Custom Local Tools**
   - Create @tool decorated functions
   - Add to tool list before agent creation

3. **Advanced Graph Patterns, based on need**
   - Use `StateGraph` for custom control flow
   - Add conditional edges for routing logic
   - Implement human-in-the-loop

4. **Persistent Memory**
   - Add checkpointer for conversation history
   - Use `MemorySaver` or Redis for state persistence

5. **Context Compression**
   - Strategy to compress the context (i.e. summarizing when context window reaches 80%)

**References:**
- LangGraph: https://langchain-ai.github.io/langgraph/
- MCP Adapters: https://reference.langchain.com/python/langchain_mcp_adapters/
- Bigdata MCP: https://docs.bigdata.com/mcp-reference/

---

## 📚 Additional Resources

- **Bigdata.com API Documentation**: https://docs.bigdata.com
- **LangGraph Documentation**: https://langchain-ai.github.io/langgraph/
- **MCP Protocol Spec**: https://modelcontextprotocol.io
- **LangSmith Tracing**: https://smith.langchain.com

**Questions?** Contact: support@bigdata.com